# 2. TF-IDF FEATURE EXTRACTION

This notebook demonstrates TF-IDF (Term Frequency-Inverse Document Frequency) feature extraction on customer feedback.

## Objectives

- Load the customer feedback dataset
- Preprocess the feedback text
- Convert text into numerical features using TF-IDF
- Examine TF-IDF feature names
- Analyze important words and phrases

In [1]:
import pandas as pd
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully.")

Libraries imported successfully.


## 2.1 Load Dataset

The customer feedback dataset is stored in `data/feedback.csv`.

We load the dataset using Pandas.

In [2]:
df = pd.read_csv("../data/feedback.csv")

print("Dataset loaded successfully.")

display(df.head())

Dataset loaded successfully.


,id,feedback,sentiment,category
0,1,Payment is failing,negative,payment
1,2,The application is very slow,negative,performance
2,3,I love the new dashboard,positive,ui
3,4,Support was very helpful,positive,support
4,5,Login OTP is not arriving,negative,login


## 2.2 Inspect Dataset

Let's examine the number of feedback records and the available columns.

In [3]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nNumber of Feedback Records:", len(df))

Dataset Shape: (30, 4)

Columns:
['id', 'feedback', 'sentiment', 'category']

Number of Feedback Records: 30


## 2.3 Text Preprocessing

Before applying TF-IDF, the customer feedback is cleaned.

The preprocessing steps include:

- Converting text to lowercase
- Removing URLs
- Removing numbers
- Removing punctuation
- Removing extra spaces

In [4]:
def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_feedback"] = df["feedback"].apply(clean_text)

display(
    df[["feedback", "clean_feedback"]].head(10)
)

,feedback,clean_feedback
0,Payment is failing,payment is failing
1,The application is very slow,the application is very slow
2,I love the new dashboard,i love the new dashboard
3,Support was very helpful,support was very helpful
4,Login OTP is not arriving,login otp is not arriving
5,The app is slow and payment fails,the app is slow and payment fails
6,The application works perfectly,the application works perfectly
7,Payment was successful,payment was successful
8,I cannot login to my account,i cannot login to my account
9,Customer support did not respond,customer support did not respond


## 2.4 Create TF-IDF Vectorizer

TF-IDF converts text into numerical values.

It gives higher importance to words that are important in a document but less common across the entire collection of documents.

We use both unigrams and bigrams to capture individual words and two-word phrases.

In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    df["clean_feedback"]
)

print("TF-IDF vectorization completed successfully.")
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF vectorization completed successfully.
TF-IDF Matrix Shape: (30, 115)


## 2.5 Extract TF-IDF Feature Names

The vectorizer creates numerical features from the words and phrases present in the customer feedback.

In [6]:
feature_names = vectorizer.get_feature_names_out()

print("Total TF-IDF Features:", len(feature_names))

print("\nFirst 30 Features:")
print(feature_names[:30])

Total TF-IDF Features: 115

First 30 Features:
['account' 'add' 'add dark' 'add fingerprint' 'app' 'app crashes'
 'app loads' 'app slow' 'application' 'application keeps'
 'application slow' 'application works' 'arriving' 'bug' 'bug application'
 'confusing' 'crashes' 'crashes frequently' 'crashing' 'customer'
 'customer support' 'dark' 'dark mode' 'dashboard' 'dashboard looks' 'did'
 'did respond' 'error' 'export' 'export feature']


## 2.6 Display TF-IDF Matrix

The TF-IDF matrix represents each customer feedback message as a numerical vector.

In [7]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

display(tfidf_df.head())

,account,add,add dark,add fingerprint,app,app crashes,app loads,app slow,application,application keeps,...,takes long,twice,want,want new,website,website takes,working,working fine,works,works perfectly
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.505187,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2.7 Identify Important Terms

The TF-IDF scores can be used to identify important words and phrases in customer feedback.

Higher TF-IDF values indicate terms that are more important within a particular feedback message.

In [8]:
first_feedback_scores = tfidf_matrix[0].toarray().flatten()

term_scores = pd.DataFrame({
    "term": feature_names,
    "tfidf_score": first_feedback_scores
})

term_scores = term_scores.sort_values(
    by="tfidf_score",
    ascending=False
)

print("Important Terms in First Feedback:")

display(term_scores.head(10))

Important Terms in First Feedback:


,term,tfidf_score
32,failing,0.639873
82,payment failing,0.639873
80,payment,0.425587
3,add fingerprint,0.000000
4,app,0.000000
5,app crashes,0.000000
6,app loads,0.000000
7,app slow,0.000000
8,application,0.000000
9,application keeps,0.000000


## 2.8 Top TF-IDF Terms for Each Feedback

Let's identify the highest-scoring terms for each customer feedback message.

In [9]:
def get_top_terms(row_index, top_n=5):

    scores = tfidf_matrix[row_index].toarray().flatten()

    ranked_indices = scores.argsort()[::-1][:top_n]

    return [
        (feature_names[i], round(scores[i], 4))
        for i in ranked_indices
        if scores[i] > 0
    ]


for i in range(min(10, len(df))):

    print(f"\nFeedback {i + 1}:")
    print(df.iloc[i]["feedback"])

    print("Top Terms:")
    print(get_top_terms(i))


Feedback 1:
Payment is failing
Top Terms:
[('payment failing', np.float64(0.6399)), ('failing', np.float64(0.6399)), ('payment', np.float64(0.4256))]

Feedback 2:
The application is very slow
Top Terms:
[('application slow', np.float64(0.6691)), ('slow', np.float64(0.5451)), ('application', np.float64(0.5052))]

Feedback 3:
I love the new dashboard
Top Terms:
[('love new', np.float64(0.4736)), ('new dashboard', np.float64(0.4736)), ('love', np.float64(0.4736)), ('dashboard', np.float64(0.4223)), ('new', np.float64(0.3858))]

Feedback 4:
Support was very helpful
Top Terms:
[('support helpful', np.float64(0.6127)), ('helpful', np.float64(0.6127)), ('support', np.float64(0.4992))]

Feedback 5:
Login OTP is not arriving
Top Terms:
[('otp arriving', np.float64(0.4826)), ('arriving', np.float64(0.4826)), ('login otp', np.float64(0.4826)), ('otp', np.float64(0.4303)), ('login', np.float64(0.3409))]

Feedback 6:
The app is slow and payment fails
Top Terms:
[('slow payment', np.float64(0.4163)

## 2.9 TF-IDF Feature Summary

The TF-IDF representation can now be used as input for machine learning models such as sentiment classification and customer feedback category classification.

In [10]:
print("TF-IDF SUMMARY")
print("=" * 50)

print("Number of Documents:", tfidf_matrix.shape[0])
print("Number of Features:", tfidf_matrix.shape[1])

print("\nExample Features:")
print(feature_names[:20])

TF-IDF SUMMARY
Number of Documents: 30
Number of Features: 115

Example Features:
['account' 'add' 'add dark' 'add fingerprint' 'app' 'app crashes'
 'app loads' 'app slow' 'application' 'application keeps'
 'application slow' 'application works' 'arriving' 'bug' 'bug application'
 'confusing' 'crashes' 'crashes frequently' 'crashing' 'customer']


## 2.10 Conclusion

TF-IDF successfully converted customer feedback into numerical feature vectors.

The process included:

1. Loading the feedback dataset
2. Cleaning the text
3. Creating TF-IDF features
4. Extracting feature names
5. Examining TF-IDF scores
6. Identifying important words and phrases

These TF-IDF features can be used for sentiment analysis and category classification.